#### Clone project and set path

In [1]:
import os

# Clone if not already cloned
if not os.path.exists('/content/medgemma-fl'):
    !git clone https://github.com/QianyuFan8/medgemma-fl.git

%cd /content/medgemma-fl
print("Current working directory:", os.getcwd())

/content/medgemma-fl
Current working directory: /content/medgemma-fl


#### Turbo multi-threaded API direct download

In [2]:
import os
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def turbo_download_gdc():
    manifest_path = "target_data/gdc_manifest_rnaseq.txt"
    output_dir = "target_data_raw/rnaseq"

    if not os.path.exists(manifest_path):
        print(f"Manifest file not found: {manifest_path}")
        return

    with open(manifest_path, 'r') as f:
        lines = f.readlines()[1:]

    file_info = [line.strip().split('\t')[:2] for line in lines if line.strip()]

    print(f"Preparing to download {len(file_info)} RNA-seq files...")

    def download_single_file(info):
        file_id, filename = info
        save_dir = os.path.join(output_dir, file_id)
        file_path = os.path.join(save_dir, filename)

        # Resume download logic
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            return

        os.makedirs(save_dir, exist_ok=True)
        url = f"https://api.gdc.cancer.gov/data/{file_id}"

        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200:
                with open(file_path, 'wb') as f:
                    f.write(r.content)
        except Exception:
            pass

    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = [executor.submit(download_single_file, item) for item in file_info]
        for _ in tqdm(as_completed(futures), total=len(file_info), desc="Turbo download progress"):
            pass

    print("\nDownload complete! All files saved to", output_dir)

turbo_download_gdc()

Preparing to download 3894 RNA-seq files...


Turbo download progress: 100%|██████████| 3894/3894 [00:00<00:00, 48432.27it/s]


Download complete! All files saved to target_data_raw/rnaseq


#### Extract and merge expression matrix with clinical survival outcomes

In [3]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm

def process_target_dataset():
    print("Starting to refine TARGET dataset...")

    BASE_DIR = os.getcwd()
    SAMPLE_SHEET_PATH = os.path.join(BASE_DIR, 'target_data', 'gdc_sample_sheet_rnaseq.tsv')
    CLINICAL_PATH = os.path.join(BASE_DIR, 'target_data', 'clinical', 'clinical.tsv')
    RAW_DIR = os.path.join(BASE_DIR, 'target_data_raw', 'rnaseq')
    PROCESSED_DIR = os.path.join(BASE_DIR, 'target_data_processed')

    os.makedirs(PROCESSED_DIR, exist_ok=True)

    def get_col(df, possible_names):
        for name in possible_names:
            for col in df.columns:
                if col.lower() == name.lower() or col.lower().endswith('.' + name.lower()):
                    return col
        return None

    # 1. Parse Sample Sheet
    print("\nParsing Sample Sheet...")
    sample_sheet = pd.read_csv(SAMPLE_SHEET_PATH, sep='\t')

    if 'Tissue Type' in sample_sheet.columns:
        tumor_samples = sample_sheet[sample_sheet['Tissue Type'].str.contains('Tumor|Primary', case=False, na=False)]
    elif 'Sample Type' in sample_sheet.columns:
        tumor_samples = sample_sheet[sample_sheet['Sample Type'].str.contains('Tumor|Primary', case=False, na=False)]
    else:
        tumor_samples = sample_sheet

    uuid_to_case = dict(zip(tumor_samples['File ID'], tumor_samples['Case ID']))
    print(f"Found {len(uuid_to_case)} valid tumor sample Case IDs.")

    # 2. Extract TPM expression
    print("\nExtracting and merging 3,894 TPM expression matrices...")
    expr_dict = {}

    for file_id, case_id in tqdm(uuid_to_case.items(), desc="Merging matrix progress"):
        folder_path = os.path.join(RAW_DIR, file_id)
        if not os.path.exists(folder_path):
            continue

        tsv_files = glob.glob(os.path.join(folder_path, "*.tsv"))
        if not tsv_files:
            continue

        file_path = tsv_files[0]
        try:
            df = pd.read_csv(file_path, sep='\t', comment='#', usecols=['gene_name', 'tpm_unstranded'])
            df = df[~df['gene_name'].astype(str).str.startswith('N_')]
            df = df.dropna(subset=['gene_name']).drop_duplicates(subset=['gene_name'])
            expr_dict[case_id] = df.set_index('gene_name')['tpm_unstranded']
        except Exception:
            continue

    expr_matrix = pd.DataFrame(expr_dict)
    min_samples = int(expr_matrix.shape[1] * 0.1)
    expr_matrix = expr_matrix[(expr_matrix > 0).sum(axis=1) >= min_samples]
    print(f"Expression matrix extracted successfully! Genes: {expr_matrix.shape[0]}, Patients: {expr_matrix.shape[1]}")

    # 3. Clean Clinical Data (fix: prioritize submitter_id to match TARGET-xx format)
    print("\nCleaning clinical survival outcomes...")
    clinical = pd.read_csv(CLINICAL_PATH, sep='\t')

    case_col = get_col(clinical, ['submitter_id', 'case_submitter_id', 'case_id'])
    vital_col = get_col(clinical, ['vital_status'])
    death_col = get_col(clinical, ['days_to_death'])
    followup_col = get_col(clinical, ['days_to_last_follow_up'])
    gender_col = get_col(clinical, ['gender'])
    age_col = get_col(clinical, ['age_at_index', 'age_at_diagnosis'])
    diag_col = get_col(clinical, ['primary_diagnosis'])
    proj_col = get_col(clinical, ['project_id'])

    print(f"Matched patient ID column name: {case_col}")

    clinical['os_event'] = clinical[vital_col].apply(lambda x: 1 if str(x).lower() == 'dead' else 0)

    days_death = pd.to_numeric(clinical[death_col], errors='coerce') if death_col else pd.Series(np.nan, index=clinical.index)
    days_followup = pd.to_numeric(clinical[followup_col], errors='coerce') if followup_col else pd.Series(np.nan, index=clinical.index)
    clinical['os_days'] = days_death.fillna(days_followup)

    if age_col:
        clinical['age_years'] = pd.to_numeric(clinical[age_col], errors='coerce')
    else:
        clinical['age_years'] = np.nan

    clinical['case_id'] = clinical[case_col]
    clinical['project_id'] = clinical[proj_col] if proj_col else 'TARGET'
    clinical['primary_diagnosis'] = clinical[diag_col] if diag_col else 'Unknown'
    clinical['gender'] = clinical[gender_col] if gender_col else 'Unknown'

    clean_cols = ['case_id', 'project_id', 'primary_diagnosis', 'gender', 'age_years', 'os_event', 'os_days']
    clinical_clean = clinical[clean_cols].drop_duplicates(subset=['case_id']).set_index('case_id')

    # 4. Align and Output
    print("\nAligning expression matrix with clinical data...")
    print("Expression matrix sample ID example:", list(expr_matrix.columns[:3]))
    print("Clinical table sample ID example:", list(clinical_clean.index[:3]))

    common_cases = list(set(expr_matrix.columns).intersection(set(clinical_clean.index)))
    print(f"\nSuccessfully aligned: {len(common_cases)} patients!")

    expr_aligned = expr_matrix[common_cases].T
    clinical_aligned = clinical_clean.loc[common_cases]
    log2_expr_aligned = np.log2(expr_aligned + 1.0)

    out_expr = os.path.join(PROCESSED_DIR, 'target_rnaseq_log2_tpm.parquet')
    out_clinical = os.path.join(PROCESSED_DIR, 'target_clinical_cleaned.csv')

    log2_expr_aligned.to_parquet(out_expr)
    clinical_aligned.to_csv(out_clinical)

    print("\nAll complete! Generated final gold standard dataset:")
    print(f"  ├── Expression Matrix (Log2-TPM): {out_expr}")
    print(f"  └── Clinical Annotation (Survival Label): {out_clinical}")

process_target_dataset()

Starting to refine TARGET dataset...

Parsing Sample Sheet...
Found 3894 valid tumor sample Case IDs.

Extracting and merging 3,894 TPM expression matrices...


Merging matrix progress: 100%|██████████| 3894/3894 [05:26<00:00, 11.93it/s]


Expression matrix extracted successfully! Genes: 49436, Patients: 3214

Cleaning clinical survival outcomes...
Matched patient ID column name: cases.submitter_id

Aligning expression matrix with clinical data...
Expression matrix sample ID example: ['TARGET-20-PAVRCS', 'TARGET-20-PAXDMP', 'TARGET-20-PAXCDU']
Clinical table sample ID example: ['TARGET-51-PAJPFB', 'TARGET-30-PASWVY', 'TARGET-20-PARPSX']

Successfully aligned: 3214 patients!

All complete! Generated final gold standard dataset:
  ├── Expression Matrix (Log2-TPM): /content/medgemma-fl/target_data_processed/target_rnaseq_log2_tpm.parquet
  └── Clinical Annotation (Survival Label): /content/medgemma-fl/target_data_processed/target_clinical_cleaned.csv


#### Preview processed datasets

In [4]:
# 1. Read the two generated files
df_expr = pd.read_parquet('target_data_processed/target_rnaseq_log2_tpm.parquet')
df_clinical = pd.read_csv('target_data_processed/target_clinical_cleaned.csv')

print("="*60)
print("【1. Clinical Phenotype and Survival Outcome Data (target_clinical_cleaned.csv)】")
print("="*60)
print("Data Dimension (patients x clinical features):")
print(df_clinical.shape)
print("\nSurvival Status Distribution (os_event: 0=alive, 1=dead):")
print(df_clinical['os_event'].value_counts())
print("\nDiagnosis Classification Distribution (Primary Diagnosis Top 5):")
print(df_clinical['primary_diagnosis'].value_counts().head(5))
print("\nPreview of the first 5 rows:")
display(df_clinical.head(5))

print("\n" + "="*60)
print("【2. RNA-seq Gene Expression Matrix (target_rnaseq_log2_tpm.parquet)】")
print("="*60)
print("Data Dimension (patients x genes):")
print(df_expr.shape)
print("\nPreview of expression values (Log2-TPM) for the first 5 patients x first 8 genes:")
display(df_expr.iloc[:5, :8])

【1. Clinical Phenotype and Survival Outcome Data (target_clinical_cleaned.csv)】
Data Dimension (patients x clinical features):
(3214, 7)

Survival Status Distribution (os_event: 0=alive, 1=dead):
os_event
0    2184
1    1030
Name: count, dtype: int64

Diagnosis Classification Distribution (Primary Diagnosis Top 5):
primary_diagnosis
Acute myeloid leukemia, NOS    1987
Acute lymphocytic leukemia      573
'--                             207
Neuroblastoma, NOS              130
Wilms tumor                     125
Name: count, dtype: int64

Preview of the first 5 rows:


,case_id,project_id,primary_diagnosis,gender,age_years,os_event,os_days
0,TARGET-20-PARLHW,TARGET-AML,"Acute myeloid leukemia, NOS",Unknown,8.0,0,3020.0
1,TARGET-20-PAVSHX,TARGET-AML,"Acute myeloid leukemia, NOS",Unknown,16.0,1,623.0
2,TARGET-20-PAXBAV,TARGET-AML,"Acute myeloid leukemia, NOS",Unknown,8.0,1,206.0
3,TARGET-10-PAPCUR,TARGET-ALL-P2,Acute lymphocytic leukemia,Unknown,3.0,1,369.0
4,TARGET-20-PAVBEM,TARGET-AML,"Acute myeloid leukemia, NOS",Unknown,4.0,0,2090.0



【2. RNA-seq Gene Expression Matrix (target_rnaseq_log2_tpm.parquet)】
Data Dimension (patients x genes):
(3214, 49436)

Preview of expression values (Log2-TPM) for the first 5 patients x first 8 genes:


gene_name,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2
TARGET-20-PARLHW,0.270230,0.000000,5.322696,1.726003,0.728922,4.941646,0.098015,3.193172
TARGET-20-PAVSHX,0.116099,0.000000,5.442632,2.325012,2.064090,6.637896,0.110096,5.850997
TARGET-20-PAXBAV,0.037031,0.000000,5.784920,3.013837,2.077346,3.980647,0.235605,3.435882
TARGET-10-PAPCUR,0.870187,0.000000,5.281353,1.222619,1.897628,5.795120,0.257252,3.341730
TARGET-20-PAVBEM,0.069565,0.105544,5.660777,2.532990,3.745280,7.202201,0.039981,3.858439
